# WP 12 · Edge Detection
*Live lab — Unit II: UAV Data Collection & Processing Methods*

An edge is a fast change in intensity - a large **gradient**. Derivative kernels estimate it; Canny then cleans it into thin, connected lines. Use `CV_64F` so negative gradients aren't clipped.

**Supporting data:** `aerial_scene.jpg` (download it from the button next to this snippet in the playground, then upload when the notebook asks).

---
**How to use:** `Runtime ▸ Run all`, or run each cell top to bottom. The original teaching snippet is reproduced verbatim below; only GUI-only calls (`cv2.imshow`, `cv2.waitKey`, `cv2.destroyAllWindows`) are adapted, because Colab has no display window.

In [ ]:
# === Setup (run me first) ===================================================
# OpenCV, NumPy and Matplotlib are already installed in Google Colab.
# If you run locally and cv2 is missing, uncomment the next line:
# !pip install opencv-python-headless matplotlib

import cv2, numpy as np, os
import matplotlib.pyplot as plt
print("OpenCV", cv2.__version__)

def show(*imgs, titles=None, cmap=None, figsize=(13, 5)):
    """Display 1..N images inline. BGR images are auto-converted to RGB.
    (Colab has no window server, so cv2.imshow() cannot be used.)"""
    titles = titles or [""] * len(imgs)
    plt.figure(figsize=figsize)
    for i, im in enumerate(imgs):
        ax = plt.subplot(1, len(imgs), i + 1)
        if im.ndim == 2:
            ax.imshow(im, cmap=cmap or "gray", vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(titles[i], fontsize=11); ax.axis("off")
    plt.tight_layout(); plt.show()


In [ ]:
# --- Stand-in scene generator (used only if you don't upload the data file) ---
def make_scene(w=768, h=512, seed=7):
    rng = np.random.default_rng(seed)
    img = np.full((h, w, 3), (78, 120, 96), np.uint8)
    yy, xx = np.mgrid[0:h, 0:w]
    img = np.clip(img + (16*np.sin(xx/130) + 10*np.cos(yy/90))[..., None], 0, 255).astype(np.uint8)
    crop = img.copy()
    for k in range(-h, w, 9):
        cv2.line(crop, (k, 0), (k + h, h), (60, 165, 90), 2, cv2.LINE_AA)
    m = np.zeros((h, w), np.uint8); cv2.rectangle(m, (0, 150), (470, h), 255, -1)
    img[m == 255] = crop[m == 255]
    cv2.ellipse(img, (628, 120), (95, 62), 18, 0, 360, (150, 92, 40), -1, cv2.LINE_AA)
    pts = np.array([[0,470],[180,430],[330,360],[430,250],[520,170],[640,90],[w,40]], np.int32)
    cv2.polylines(img, [pts], False, (70, 72, 78), 26, cv2.LINE_AA)
    for cx, cy, bw, bh, col, a in [(150,250,84,60,(205,205,210),8),(250,300,70,52,(120,150,225),-6),
                                   (360,190,60,44,(225,225,230),20),(120,360,56,40,(150,175,235),4)]:
        r = cv2.boxPoints(((cx,cy),(bw,bh),a)).astype(np.int32)
        cv2.fillConvexPoly(img, r, col, cv2.LINE_AA); cv2.polylines(img, [r], True, (40,40,45), 2, cv2.LINE_AA)
    for cx, cy, rr in [(60,120,16),(95,175,13),(300,110,15),(430,430,18),(500,470,14),(700,300,17),(610,380,13),(250,460,15)]:
        cv2.circle(img, (cx, cy), rr, (40, 95, 45), -1, cv2.LINE_AA)
    for cx, cy, col, a in [(300,380,(250,250,250),-32),(470,205,(60,60,235),40)]:
        r = cv2.boxPoints(((cx,cy),(26,12),a)).astype(np.int32); cv2.fillConvexPoly(img, r, col, cv2.LINE_AA)
    return img

def get_image(name, gray=False):
    """Load `name` if present; else offer a Colab upload; else auto-generate."""
    flag = cv2.IMREAD_UNCHANGED if name.lower().endswith(".png") else cv2.IMREAD_COLOR
    if os.path.exists(name):
        im = cv2.imread(name, flag)
        if im is not None:
            print("Loaded", name)
            return cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) if gray else im
    try:
        from google.colab import files
        print(f"Upload '{name}' from the data pack (or press Cancel to auto-generate).")
        up = files.upload()
        for fn in up:
            im = cv2.imread(fn, flag)
            if im is not None:
                print("Loaded", fn)
                return cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) if gray else im
    except Exception:
        pass
    print("Auto-generating a stand-in scene.")
    s = make_scene()
    return cv2.cvtColor(s, cv2.COLOR_BGR2GRAY) if gray else s


### The snippet, exactly as shown in the playground
```python
gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
lap = cv2.Laplacian(gray, cv2.CV_64F)      # 2nd derivative

edges = cv2.Canny(gray, 40, 100)           # low, high
```

In [ ]:
gray = get_image("aerial_scene.jpg", gray=True)

gx  = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)   # vertical edges
gy  = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)   # horizontal edges
lap = cv2.Laplacian(gray, cv2.CV_64F)              # 2nd derivative

mag = cv2.magnitude(gx, gy)
mag = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

edges = cv2.Canny(gray, 40, 100)                   # low, high

show(gray, mag, edges, titles=["Grayscale", "Sobel gradient magnitude", "Canny edges"])

In [ ]:
# --- Why the threshold ratio matters (Canny advises ~2:1 to 3:1) ------------
show(cv2.Canny(gray, 20, 40),
     cv2.Canny(gray, 40, 100),
     cv2.Canny(gray, 100, 200),
     titles=["20/40 - noisy", "40/100 - balanced", "100/200 - sparse"])
pct = 100 * (cv2.Canny(gray, 40, 100) > 0).mean()
print(f"Edge pixels at 40/100: {pct:.1f}% of the frame")